# Case Study 1: News Article Search & Recommendation Engine

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TechyNilesh/DeepTextSearch/blob/main/notebooks/01_case_study_news_search.ipynb)

**Goal**: Build a news article search engine that supports:
- Hybrid search (semantic + keyword)
- Category filtering via metadata
- Cross-encoder reranking for precision
- Saving and loading the index

**Features covered**: `TextEmbedder`, `TextSearch`, `Reranker`, hybrid/dense/bm25 modes, metadata filtering, save/load, incremental indexing

In [1]:
import nest_asyncio
nest_asyncio.apply()

In [ ]:
!pip install -q git+https://github.com/TechyNilesh/DeepTextSearch.git

## Step 1: Prepare the News Dataset

We'll create a realistic news corpus with categories and dates.

In [3]:
import pandas as pd

news_data = {
    "title": [
        "OpenAI Releases GPT-5 with Reasoning Capabilities",
        "Tesla Stock Surges After Record Deliveries",
        "SpaceX Starship Completes First Orbital Flight",
        "New Study Shows Exercise Reduces Dementia Risk by 40%",
        "India Wins Cricket World Cup in Dramatic Final",
        "EU Passes Landmark AI Regulation Act",
        "Google DeepMind Achieves Breakthrough in Protein Folding",
        "Federal Reserve Holds Interest Rates Steady",
        "Apple Launches Vision Pro 2 with Enhanced AR",
        "Climate Summit Reaches Historic Carbon Reduction Agreement",
        "NVIDIA Reports Record Revenue from AI Chip Demand",
        "WHO Declares End of Global Mpox Emergency",
        "Microsoft Acquires Cybersecurity Startup for $2 Billion",
        "Olympics 2028: Los Angeles Unveils New Venues",
        "Meta Open-Sources Llama 4 Language Model",
        "Amazon Expands Drone Delivery to 10 New Cities",
        "Breakthrough Battery Technology Doubles EV Range",
        "China Lands Rover on Far Side of the Moon Again",
        "Major Data Breach Affects 50 Million Healthcare Records",
        "Quantum Computer Solves Problem Impossible for Classical Machines",
    ],
    "content": [
        "OpenAI has released GPT-5, its most advanced language model yet, featuring improved reasoning capabilities and the ability to solve multi-step problems with greater accuracy.",
        "Tesla shares jumped 12% after the company reported record quarterly deliveries of 500,000 vehicles, beating analyst expectations across all regions.",
        "SpaceX successfully completed the first orbital flight of its Starship mega-rocket, marking a major milestone in the company's plans for Mars colonization.",
        "A large-scale study published in The Lancet found that regular physical exercise can reduce the risk of developing dementia by up to 40% in adults over 50.",
        "India secured a thrilling victory in the Cricket World Cup final, chasing down a record target of 350 runs in the last over against Australia at Lord's.",
        "The European Union has passed comprehensive AI regulation requiring transparency, risk assessment, and human oversight for high-risk artificial intelligence systems.",
        "Google DeepMind's AlphaFold 3 has achieved unprecedented accuracy in predicting protein structures, potentially revolutionizing drug discovery and molecular biology.",
        "The Federal Reserve decided to hold interest rates steady at 5.25%, citing mixed economic signals and ongoing inflation concerns in the housing market.",
        "Apple unveiled Vision Pro 2 with improved augmented reality features, lighter design, and a new spatial computing platform for enterprise applications.",
        "World leaders at the COP30 climate summit agreed to a historic carbon reduction framework, pledging to cut emissions by 60% by 2035.",
        "NVIDIA reported quarterly revenue of $35 billion, driven by explosive demand for its AI training and inference GPUs from cloud providers and enterprises.",
        "The World Health Organization officially declared the end of the global mpox public health emergency after successful vaccination campaigns worldwide.",
        "Microsoft acquired AI-powered cybersecurity firm CyberShield for $2 billion to strengthen its cloud security offerings and threat detection capabilities.",
        "The 2028 Los Angeles Olympics organizing committee revealed plans for five new state-of-the-art venues designed for sustainability and accessibility.",
        "Meta released Llama 4 as open-source, featuring 400 billion parameters, multimodal capabilities, and state-of-the-art performance on reasoning benchmarks.",
        "Amazon announced expansion of its Prime Air drone delivery service to 10 additional US cities, promising 30-minute deliveries for eligible packages.",
        "Researchers at Stanford have developed a solid-state battery technology that doubles electric vehicle range to over 600 miles on a single charge.",
        "China's Chang'e-7 mission successfully landed a rover on the far side of the Moon, collecting samples from a previously unexplored crater.",
        "A massive cyberattack on a major US healthcare system exposed personal and medical records of 50 million patients, raising concerns about data security.",
        "IBM's latest quantum processor solved a complex optimization problem in 4 minutes that would take the fastest classical supercomputer over 10,000 years.",
    ],
    "category": [
        "Technology", "Business", "Space", "Health", "Sports",
        "Politics", "Technology", "Business", "Technology", "Politics",
        "Business", "Health", "Technology", "Sports", "Technology",
        "Technology", "Technology", "Space", "Cybersecurity", "Technology",
    ],
    "year": [
        2025, 2025, 2025, 2025, 2025,
        2025, 2025, 2025, 2026, 2026,
        2026, 2026, 2026, 2026, 2026,
        2026, 2026, 2026, 2026, 2026,
    ],
}

df = pd.DataFrame(news_data)
print(f"Dataset: {len(df)} articles")
print(f"Categories: {df['category'].value_counts().to_dict()}")
df.head()

Dataset: 20 articles
Categories: {'Technology': 8, 'Business': 3, 'Space': 2, 'Health': 2, 'Sports': 2, 'Politics': 2, 'Cybersecurity': 1}


,title,content,category,year
0,OpenAI Releases GPT-5 with Reasoning Capabilities,"OpenAI has released GPT-5, its most advanced l...",Technology,2025
1,Tesla Stock Surges After Record Deliveries,Tesla shares jumped 12% after the company repo...,Business,2025
2,SpaceX Starship Completes First Orbital Flight,SpaceX successfully completed the first orbita...,Space,2025
3,New Study Shows Exercise Reduces Dementia Risk...,A large-scale study published in The Lancet fo...,Health,2025
4,India Wins Cricket World Cup in Dramatic Final,India secured a thrilling victory in the Crick...,Sports,2025


## Step 2: Index with Metadata

In [4]:
from DeepTextSearch import TextEmbedder, TextSearch

embedder = TextEmbedder(model_name="BAAI/bge-small-en-v1.5")
embedder.index(
    df,
    text_column="content",
    metadata_columns=["title", "category", "year"],
)

print(f"Indexed {embedder.corpus_size} articles ({embedder.dimension}d embeddings)")
print(f"Device: {embedder.device}")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Indexed 20 articles (384d embeddings)
Device: cuda


## Step 3: Hybrid Search

In [5]:
search = TextSearch(embedder, mode="hybrid")

query = "latest AI language model releases"
results = search.search(query, top_n=5)

print(f"Query: '{query}'\n")
for i, r in enumerate(results, 1):
    print(f"{i}. [{r.score:.6f}] [{r.metadata['category']}] {r.metadata['title']}")
    print(f"   {r.text[:100]}...\n")

Query: 'latest AI language model releases'

1. [0.016393] [Technology] OpenAI Releases GPT-5 with Reasoning Capabilities
   OpenAI has released GPT-5, its most advanced language model yet, featuring improved reasoning capabi...

2. [0.015724] [Politics] EU Passes Landmark AI Regulation Act
   The European Union has passed comprehensive AI regulation requiring transparency, risk assessment, a...

3. [0.015682] [Technology] Quantum Computer Solves Problem Impossible for Classical Machines
   IBM's latest quantum processor solved a complex optimization problem in 4 minutes that would take th...

4. [0.015205] [Business] NVIDIA Reports Record Revenue from AI Chip Demand
   NVIDIA reported quarterly revenue of $35 billion, driven by explosive demand for its AI training and...

5. [0.009677] [Technology] Google DeepMind Achieves Breakthrough in Protein Folding
   Google DeepMind's AlphaFold 3 has achieved unprecedented accuracy in predicting protein structures, ...



## Step 4: Compare Search Modes

See how hybrid, dense, and BM25 differ on the same query.

In [6]:
query = "quantum computing breakthrough"

for mode in ["hybrid", "dense", "bm25"]:
    print(f"=== {mode.upper()} ===")
    for r in search.search(query, top_n=3, mode=mode):
        print(f"  [{r.score:.6f}] {r.metadata['title']}")
    print()

=== HYBRID ===
  [0.016288] Quantum Computer Solves Problem Impossible for Classical Machines
  [0.015513] Apple Launches Vision Pro 2 with Enhanced AR
  [0.009677] Google DeepMind Achieves Breakthrough in Protein Folding

=== DENSE ===
  [0.737900] Quantum Computer Solves Problem Impossible for Classical Machines
  [0.603204] Google DeepMind Achieves Breakthrough in Protein Folding
  [0.581199] OpenAI Releases GPT-5 with Reasoning Capabilities

=== BM25 ===
  [2.600077] Apple Launches Vision Pro 2 with Enhanced AR
  [2.546425] Quantum Computer Solves Problem Impossible for Classical Machines



## Step 5: Metadata Filtering

Search within specific categories or years.

In [7]:
# Only Technology articles
print("=== Technology Only ===")
results = search.search(
    "AI breakthroughs",
    top_n=5,
    filter_fn=lambda text, meta: meta.get("category") == "Technology",
)
for r in results:
    print(f"  [{r.metadata['category']}] {r.metadata['title']}")

# Only 2026 articles
print("\n=== Year 2026 Only ===")
results = search.search(
    "company earnings revenue",
    top_n=5,
    filter_fn=lambda text, meta: meta.get("year") == 2026,
)
for r in results:
    print(f"  [{r.metadata['year']}] {r.metadata['title']}")

# Combined: Technology + 2026
print("\n=== Technology + 2026 ===")
results = search.search(
    "open source AI",
    top_n=5,
    filter_fn=lambda text, meta: meta.get("category") == "Technology" and meta.get("year") == 2026,
)
for r in results:
    print(f"  [{r.metadata['category']}/{r.metadata['year']}] {r.metadata['title']}")

=== Technology Only ===
  [Technology] OpenAI Releases GPT-5 with Reasoning Capabilities
  [Technology] Google DeepMind Achieves Breakthrough in Protein Folding
  [Technology] Quantum Computer Solves Problem Impossible for Classical Machines
  [Technology] Microsoft Acquires Cybersecurity Startup for $2 Billion
  [Technology] Apple Launches Vision Pro 2 with Enhanced AR

=== Year 2026 Only ===
  [2026] NVIDIA Reports Record Revenue from AI Chip Demand
  [2026] Microsoft Acquires Cybersecurity Startup for $2 Billion
  [2026] Apple Launches Vision Pro 2 with Enhanced AR
  [2026] Meta Open-Sources Llama 4 Language Model
  [2026] Breakthrough Battery Technology Doubles EV Range

=== Technology + 2026 ===
  [Technology/2026] Meta Open-Sources Llama 4 Language Model
  [Technology/2026] Quantum Computer Solves Problem Impossible for Classical Machines
  [Technology/2026] Microsoft Acquires Cybersecurity Startup for $2 Billion
  [Technology/2026] Apple Launches Vision Pro 2 with Enhanced AR
  

## Step 6: Cross-Encoder Reranking

Retrieve broadly, then rerank for precision.

In [8]:
from DeepTextSearch import Reranker

reranker = Reranker(model_name="cross-encoder/ms-marco-MiniLM-L-6-v2")

query = "government regulation of artificial intelligence"

# Retrieve 10 candidates
candidates = search.search(query, top_n=10)
print(f"=== Before Reranking (top 5 of {len(candidates)}) ===")
for i, r in enumerate(candidates[:5], 1):
    print(f"  {i}. [{r.score:.6f}] {r.metadata['title']}")

# Rerank to top 5
reranked = reranker.rerank_search_results(query, candidates, top_n=5)
print(f"\n=== After Reranking (top 5) ===")
for i, r in enumerate(reranked, 1):
    print(f"  {i}. [{r['score']:.4f}] {r['metadata']['title']}")

=== Before Reranking (top 5 of 10) ===
  1. [0.016393] EU Passes Landmark AI Regulation Act
  2. [0.014925] Major Data Breach Affects 50 Million Healthcare Records
  3. [0.014706] NVIDIA Reports Record Revenue from AI Chip Demand
  4. [0.014560] WHO Declares End of Global Mpox Emergency
  5. [0.014061] SpaceX Starship Completes First Orbital Flight

=== After Reranking (top 5) ===
  1. [6.3282] EU Passes Landmark AI Regulation Act
  2. [-11.1631] India Wins Cricket World Cup in Dramatic Final
  3. [-11.1944] New Study Shows Exercise Reduces Dementia Risk by 40%
  4. [-11.2339] Amazon Expands Drone Delivery to 10 New Cities
  5. [-11.2390] WHO Declares End of Global Mpox Emergency


## Step 7: Incremental Indexing

Add breaking news without re-indexing everything.

In [9]:
print(f"Corpus size before: {embedder.corpus_size}")

# Breaking news arrives!
embedder.add(
    [
        "Anthropic releases Claude 5 with advanced agentic capabilities and real-time web browsing support.",
        "Global stock markets crash as trade war tensions escalate between US and China.",
    ],
    metadata=[
        {"title": "Anthropic Launches Claude 5", "category": "Technology", "year": 2026},
        {"title": "Markets Crash on Trade War Fears", "category": "Business", "year": 2026},
    ],
)

print(f"Corpus size after: {embedder.corpus_size}")

# Search now includes new articles
search_updated = TextSearch(embedder)
for r in search_updated.search("Claude AI model", top_n=3):
    print(f"  [{r.score:.6f}] {r.metadata.get('title', r.text[:50])}")

Corpus size before: 20
Corpus size after: 22
  [0.016393] Anthropic Launches Claude 5
  [0.016129] OpenAI Releases GPT-5 with Reasoning Capabilities
  [0.015873] EU Passes Landmark AI Regulation Act


## Step 8: Save & Load Index

In [10]:
import os

# Save
embedder.save("./news_search_index")
print(f"Saved to ./news_search_index/")
print(f"Files: {os.listdir('./news_search_index')}")

# Load in a fresh session
loaded = TextEmbedder.load("./news_search_index")
search_loaded = TextSearch(loaded)

print(f"\nLoaded: {loaded.corpus_size} articles")
for r in search_loaded.search("space exploration mission", top_n=3):
    print(f"  [{r.score:.6f}] {r.text[:80]}...")

Saved to ./news_search_index/
Files: ['index.faiss', 'store_meta.json', 'corpus.json', 'config.json']



Loaded: 22 articles
  [0.016235] China's Chang'e-7 mission successfully landed a rover on the far side of the Moo...
  [0.009836] SpaceX successfully completed the first orbital flight of its Starship mega-rock...
  [0.009524] The 2028 Los Angeles Olympics organizing committee revealed plans for five new s...


---

**Summary**: This case study demonstrated building a complete news search engine with hybrid search, metadata filtering, reranking, incremental updates, and persistent storage — all in under 50 lines of code.